# How to use [ReMatching](https://arxiv.org/abs/2305.09274]) to compute a functional map?

In [17]:
import gsops.backend as gs

from geomfum.dataset import NotebooksDataset
from geomfum.shape import TriangleMesh
from geomfum.shape.hierarchical import HierarchicalMesh
import polyscope as ps

[Load meshes](00_load_mesh_from_file.ipynb).

In [20]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("cat-00"))

# alternatively put you mesh path

# mesh_a = TriangleMesh.from_file("../path/to/your/mesh")

(mesh_a.n_vertices, mesh_a.n_faces)

ps.init()
ps_mesh = ps.register_surface_mesh("mesh_a", mesh_a.vertices, mesh_a.faces)
ps.show()
ps.remove_all_structures()

Create [hierarchical meshes](./11_hierarchical_mesh.ipynb).

In [ ]:
hmesh_a = HierarchicalMesh.from_registry(mesh_a, min_n_samples=200)

ps_hmesh = ps.register_surface_mesh("hmesh_a", hmesh_a.low.vertices, hmesh_a.low.faces)
ps.show()
ps.remove_all_structures()

Having the two resolution, we can compute any geometric quantity on the low resolution mesh and reproject it on the high resolution mesh.
For example:\
Scalars: as LBO basis\
Operators: as the Laplacian \
Distances: Geodesic distance matrix 


In [ ]:
hmesh_a.low.laplacian.find_spectrum(spectrum_size=200, set_as_basis=True)
hmesh_a.low.basis.use_k = 5

ps_hmesh_low = ps.register_surface_mesh(
    "hmesh_a_low", hmesh_a.low.vertices, hmesh_a.low.faces
)
ps_hmesh_low.add_scalar_quantity(
    "low_basis_1", hmesh_a.low.basis.vecs[:, 4], defined_on="vertices", cmap="bwr"
)
ps.show()
ps.remove_all_structures()

RuntimeError: [polyscope]  [EXCEPTION] unrecognized colormap name: bwr

: 

In [16]:
hmesh_a.extend_basis(set_as_basis=True)


array([[ 1.78802240e+00, -1.71259749e+00, -1.13458855e+00, ...,
        -2.96791128e-01, -2.57488754e-08,  7.38528177e-10],
       [ 1.78802240e+00, -1.79811769e+00, -9.10340171e-01, ...,
         4.15146565e-02, -2.03262591e-08,  4.09274770e-10],
       [ 1.78802240e+00, -1.80262930e+00, -8.68997569e-01, ...,
         1.43913747e-01, -1.39656896e-08,  2.66491965e-10],
       ...,
       [ 1.78802240e+00, -5.44723210e+00,  7.27501728e+00, ...,
        -9.32900966e-01,  1.11162637e-10,  2.65491610e-13],
       [ 1.78802240e+00, -5.45301973e+00,  7.28943052e+00, ...,
        -1.05048367e+00,  1.18458643e-10,  2.68934862e-13],
       [ 1.78802240e+00, -1.69923871e+00, -1.11446081e+00, ...,
        -2.51183403e-01,  6.38996970e-09,  7.21851627e-10]],
      shape=(7207, 200))

Now, this functional map can be seamlessly used with the high-resolution meshes. For example, we can [upsample it with ZoomOut](./15_refine_functional_map.ipynb).

In [7]:
upsampler = ZoomOut(nit=2, step=(2, 1))

upsampled_fmap_matrix = upsampler(
    fmap_matrix,
    hmesh_a.high.basis,
    hmesh_b.high.basis,
)

upsampled_fmap_matrix.shape

(7, 10)

In [8]:
hmesh_a.high.basis.vecs

array([[-1.72157991,  1.63091261, -1.05690336, ...,  2.67648666,
        -1.04742672, -0.71500848],
       [-1.72157991,  1.70083536, -0.86500711, ...,  2.84924518,
        -1.43599688, -0.51465914],
       [-1.72157991,  1.69559836, -0.84127265, ...,  2.87933207,
        -1.4351918 , -0.82635609],
       ...,
       [-1.72157991,  5.0574198 ,  6.82858863, ..., -3.77463283,
         6.36453632, -0.21060378],
       [-1.72157991,  5.06379084,  6.84460851, ..., -3.8108807 ,
         6.45896302, -0.21374222],
       [-1.72157991,  1.63753198, -1.00536459, ...,  2.71484979,
        -1.10020252, -1.01029031]], shape=(7207, 10))

NB: `mesh_a` and `hmesh_a.high` are the same object, so it is indiferent which one to use.

## Further reading

* [How to compute a functional map?](./functional_map.ipynb)

* [How to refine a functional map?](./15_refine_functional_map.ipynb)